# Touchstone I/O: load `.s1p`, plot S11 dB + Smith

Round-trip a single-port Touchstone file through the `yee` Python bindings and produce the three plots an RF engineer expects: `|S11|` in dB vs frequency, the reflection coefficient on a Smith chart, and the derived input impedance `Z_in(f)`.

After a solver run, results land in a Touchstone file. The Python bindings let you load it back, run downstream analysis, and plot. This notebook reads a synthetic resonant dipole fixture (`examples/data/dipole_synth.s1p`) shipped with the repo so it runs without a prior solve.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import yee
from yee import s11_db, smith_xy

# `yee.touchstone` is exposed as an attribute on the top-level `yee` module
# (a PyO3 submodule object), so reach it as `yee.touchstone.read` rather
# than `from yee.touchstone import read` — the latter requires the
# submodule to be registered in `sys.modules`, which PyO3 doesn't do
# automatically.
read_touchstone = yee.touchstone.read

In [ ]:
# Locate the fixture relative to this notebook. In Jupyter `__file__`
# isn't defined, so fall back to the current working directory; in
# either case we walk up to the examples/ root and dip into data/.
_here = os.path.dirname(os.path.realpath(__file__)) if "__file__" in globals() else os.getcwd()
fixture = os.path.join(_here, "..", "data", "dipole_synth.s1p")
ts = read_touchstone(fixture)

freq = ts["freq_hz"]
s = ts["data"]    # shape (F, N, N) complex
s11 = s[:, 0, 0]
print(f"n_ports:    {ts['n_ports']}")
print(f"n_freqs:    {len(freq)}")
print(f"z0:         {ts['z0']}")
print(f"freq range: {freq.min()/1e9:.2f} - {freq.max()/1e9:.2f} GHz")

Plot `|S11|` in dB.

In [ ]:
db = s11_db(s11)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(freq / 1e9, db, marker="o")
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("|S11| (dB)")
ax.set_title("Synthetic dipole S11")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Smith chart (S11 reflection coefficient on the complex unit-disk). `yee.smith_xy` returns an `(N, 2)` array; column 0 is `Re(S11)`, column 1 is `Im(S11)`.

In [ ]:
xy = smith_xy(s11)
x = xy[:, 0]
y = xy[:, 1]
fig, ax = plt.subplots(figsize=(5, 5))
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), "k-", lw=1, alpha=0.4)
ax.plot([-1, 1], [0, 0], "k-", lw=1, alpha=0.4)
ax.plot(x, y, marker="o", label="S11(freq)")
mid = len(x) // 2
ax.scatter(x[mid], y[mid], color="red", zorder=5,
           label=f"f={freq[mid]/1e9:.2f} GHz")
ax.set_aspect("equal", "box")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("Re(S11)")
ax.set_ylabel("Im(S11)")
ax.set_title("Smith chart")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

Convert `S11` to input impedance: `Z = Z0 * (1 + S11) / (1 - S11)`.

In [ ]:
z0 = ts["z0"]
z_in = z0 * (1.0 + s11) / (1.0 - s11)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(freq / 1e9, z_in.real, label="Re(Z)")
ax.plot(freq / 1e9, z_in.imag, label="Im(Z)")
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Z_in (\u03a9)")
ax.set_title("Input impedance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

To round-trip your own data back to disk, see `yee.touchstone.write(path, file_dict)` — the same dict schema this notebook consumed from `read`.